In [1]:
"""
Build the stop-level graph used by the STGCN model.

Nodes  = every stop_id that appears in segment_network (i.e. every stop
         actually served by a trip in your filtered route set).
Edges  = (1) route edges, taken directly from segment_network
             (directed, weight = inverse scheduled travel time)
         (2) spatial edges, stops within SPATIAL_RADIUS_M of each other
             that are NOT already a route edge in either direction
             (undirected, weight = Gaussian kernel of distance)

Spatial edges are the part your current pipeline doesn't have: they're
what let the graph see "M1 and M15 both sit on 3rd Ave here" even
though segment_network gives them different segment_ids.

Outputs:
  processed/graph_nodes.parquet   node_idx, stop_id, stop_lat, stop_lon
  processed/graph_edges.parquet   src_idx, dst_idx, weight, edge_type
"""
import math
from pathlib import Path

import numpy as np
import polars as pl
from scipy.spatial import cKDTree

PROCESSED = Path("processed")
SPATIAL_RADIUS_M = 175.0   # ~ one NYC short block; tune per city block size
SPATIAL_SIGMA_M = 100.0    # Gaussian kernel width for spatial edge weight

segment_network = pl.read_parquet(PROCESSED / "segment_network.parquet")

# ---------------------------------------------------------------------
# 1. Node table — one row per distinct stop_id, keep a representative
#    lat/lon (segment_network already carries start/end coords per row)
# ---------------------------------------------------------------------
starts = segment_network.select([
    pl.col("stop_id"),
    pl.col("start_lat").alias("stop_lat"),
    pl.col("start_lon").alias("stop_lon"),
])
ends = segment_network.select([
    pl.col("next_stop_id").alias("stop_id"),
    pl.col("end_lat").alias("stop_lat"),
    pl.col("end_lon").alias("stop_lon"),
])
nodes = (
    pl.concat([starts, ends])
    .unique(subset=["stop_id"])
    .sort("stop_id")
    .with_row_index("node_idx")
)
nodes.write_parquet(PROCESSED / "graph_nodes.parquet")
print(f"nodes: {nodes.height}")

stop_to_idx = dict(zip(nodes["stop_id"].to_list(), nodes["node_idx"].to_list()))

# ---------------------------------------------------------------------
# 2. Route edges — collapse across route/direction/shape: keep the
#    fastest observed scheduled travel time per physical stop pair as
#    the edge weight basis (a stop pair can be served by several
#    routes/shapes; we just need one graph edge, weighted well).
# ---------------------------------------------------------------------
route_edges = (
    segment_network
    .filter(
        pl.col("stop_id").is_in(list(stop_to_idx)) &
        pl.col("next_stop_id").is_in(list(stop_to_idx))
    )
    .group_by(["stop_id", "next_stop_id"])
    .agg(pl.col("scheduled_travel_time").median().alias("travel_time"))
    .with_columns(
        pl.col("stop_id").replace_strict(stop_to_idx).alias("src_idx"),
        pl.col("next_stop_id").replace_strict(stop_to_idx).alias("dst_idx"),
        # inverse travel time -> shorter/faster hops get a stronger edge
        (1.0 / pl.col("travel_time").clip(lower_bound=1.0)).alias("weight"),
    )
    .select(["src_idx", "dst_idx", "weight"])
    .with_columns(pl.lit("route").alias("edge_type"))
)
print(f"route edges: {route_edges.height}")

# ---------------------------------------------------------------------
# 3. Spatial edges — KD-tree radius query in a local equirectangular
#    projection (fine at city scale; avoids a slower haversine loop).
# ---------------------------------------------------------------------
lats = nodes["stop_lat"].to_numpy()
lons = nodes["stop_lon"].to_numpy()
lat0 = float(np.mean(lats))
R = 6371000.0
lat0_rad = math.radians(lat0)
x = np.radians(lons) * R * math.cos(lat0_rad)
y = np.radians(lats) * R
coords = np.stack([x, y], axis=1)

tree = cKDTree(coords)
pairs = tree.query_pairs(r=SPATIAL_RADIUS_M, output_type="ndarray")

existing_route_pairs = set(
    zip(route_edges["src_idx"].to_list(), route_edges["dst_idx"].to_list())
)

spatial_rows = []
for i, j in pairs:
    i, j = int(i), int(j)
    if (i, j) in existing_route_pairs or (j, i) in existing_route_pairs:
        continue  # already connected structurally, no need to double up
    d = float(np.linalg.norm(coords[i] - coords[j]))
    w = math.exp(-(d ** 2) / (2 * SPATIAL_SIGMA_M ** 2))
    spatial_rows.append((i, j, w))
    spatial_rows.append((j, i, w))  # symmetric

spatial_edges = pl.DataFrame(
    spatial_rows, schema=["src_idx", "dst_idx", "weight"], orient="row"
).with_columns(pl.lit("spatial").alias("edge_type"))
print(f"spatial edges: {spatial_edges.height}")

edges = pl.concat([route_edges, spatial_edges])
edges.write_parquet(PROCESSED / "graph_edges.parquet")
print(f"total edges: {edges.height}")
print(
    "edge_type breakdown:\n",
    edges.group_by("edge_type").agg(pl.len()).sort("edge_type"),
)

nodes: 585
route edges: 679
spatial edges: 770
total edges: 1449
edge_type breakdown:
 shape: (2, 2)
┌───────────┬─────┐
│ edge_type ┆ len │
│ ---       ┆ --- │
│ str       ┆ u32 │
╞═══════════╪═════╡
│ route     ┆ 679 │
│ spatial   ┆ 770 │
└───────────┴─────┘


In [2]:
"""
Turn the irregular per-(trip, stop) event rows from build_dataset_chunked
into a regular (T, N, F) grid the STGCN can consume.

REQUIRED CHANGE upstream: your current `model_ready` only keeps
ID_COLS = ["trip_id", "start_date", "stop_id"] plus engineered features
like `hour` — there's no raw timestamp left to bin by 5 minutes. Add the
actual event epoch to ID_COLS in build_dataset_chunked.ipynb:

    ID_COLS = ["trip_id", "start_date", "stop_id", "arrival_time"]

(arrival_time is already computed as an epoch-seconds column inside
build_day — it's just getting dropped at the `model_ready = ...` select.)
Re-run that notebook once with the extra column before using this script.

Output: processed/spatiotemporal/{X.npy, mask.npy, timestamps.npy}
  X          float32 (T, N, F)  -- [speed_mps, delay_seconds, vehicle_count]
  mask       float32 (T, N)     -- 1.0 where a bin had a real observation
  timestamps int64   (T,)       -- bin start, epoch seconds
Node order matches processed/graph_nodes.parquet's node_idx.
"""
from pathlib import Path

import numpy as np
import polars as pl

GTFS = Path("raw/processed_gtfs")
PROCESSED = Path("processed")
OUT = PROCESSED / "spatiotemporal"
OUT.mkdir(parents=True, exist_ok=True)

BIN_SECONDS = 300          # 5-minute bins, standard for STGCN/DCRNN-style models
MAX_FILL_GAP_BINS = 3      # forward-fill across at most 15 min of silence
FEATURES = ["speed_mps", "delay_seconds", "vehicle_count"]

nodes = pl.read_parquet(PROCESSED / "graph_nodes.parquet")
n_nodes = nodes.height
stop_to_idx = dict(zip(nodes["stop_id"].to_list(), nodes["node_idx"].to_list()))

df = pl.read_parquet(GTFS / "baseline_dataset.parquet")
assert "arrival_time" in df.columns, (
    "baseline_dataset.parquet is missing `arrival_time` — see the module "
    "docstring: add it to ID_COLS in build_dataset_chunked.ipynb and rerun."
)

df = df.filter(pl.col("stop_id").is_in(list(stop_to_idx)))
df = df.with_columns(
    pl.col("stop_id").replace_strict(stop_to_idx).alias("node_idx"),
    (pl.col("arrival_time") // BIN_SECONDS * BIN_SECONDS).alias("time_bin"),
    # travel_time is realized minutes for the *next* segment; compare
    # against its own scheduled time as a simple, per-row delay signal
    (pl.col("travel_time") - pl.col("scheduled_segment_time")).alias("delay_seconds"),
)

agg = (
    df.group_by(["node_idx", "time_bin"])
    .agg([
        pl.col("speed_mps").mean(),
        pl.col("delay_seconds").mean(),
        pl.len().alias("vehicle_count"),
    ])
    .sort(["time_bin", "node_idx"])
)

t_min, t_max = int(agg["time_bin"].min()), int(agg["time_bin"].max())
timestamps = np.arange(t_min, t_max + BIN_SECONDS, BIN_SECONDS, dtype=np.int64)
T = len(timestamps)
t_to_row = {t: i for i, t in enumerate(timestamps)}

X = np.full((T, n_nodes, len(FEATURES)), np.nan, dtype=np.float32)
rows = agg["time_bin"].to_numpy()
cols = agg["node_idx"].to_numpy()
row_idx = np.array([t_to_row[t] for t in rows])
for f_i, f in enumerate(FEATURES):
    X[row_idx, cols, f_i] = agg[f].to_numpy()

observed_mask = (~np.isnan(X[:, :, 0])).astype(np.float32)  # speed present == observed

# forward-fill each node's series up to MAX_FILL_GAP_BINS, else leave NaN
# (NaNs get zero-filled at the very end and are excluded from loss via mask)
for n in range(n_nodes):
    last_valid = None
    gap = 0
    for t in range(T):
        if observed_mask[t, n]:
            last_valid = X[t, n, :].copy()
            gap = 0
        elif last_valid is not None:
            gap += 1
            if gap <= MAX_FILL_GAP_BINS:
                X[t, n, :] = last_valid

X = np.nan_to_num(X, nan=0.0)

np.save(OUT / "X.npy", X)
np.save(OUT / "mask.npy", observed_mask)
np.save(OUT / "timestamps.npy", timestamps)

print(f"tensor shape: {X.shape} (T, N, F)")
print(f"overall observed fraction (pre-fill): {observed_mask.mean():.3f}")

InvalidOperationError: 'is_in' cannot check for List(Int64) values in String data